# Teks — Baseline klasifikasi k-fold + OOF (jalur submission pertama)
**Kapan dipakai:** klasifikasi teks apa pun (biner / multikelas, satu atau beberapa kolom teks)
sebagai **submission pertama ≤ 45 menit**. Hasilnya juga bahan `ensemble_oof`: probabilitas OOF
tiap model disimpan ke `OOF_FOLDER`.

**Alur:** Konfigurasi → baca → bersihkan → **Temuan** (kelas, kembar, bocor train↔test,
adversarial) → k-fold → model (TF-IDF kata + karakter + SVM terkalibrasi) → pengali kelas untuk
F1 makro → simpan OOF → `submission.csv` → validator.

**Saat lomba yang diubah:** sel Konfigurasi (`# GANTI saat lomba`), lalu Run All. Baca sel Temuan,
putuskan split (Stratified / Group / waktu) dan pembersihan, ubah `MODEL_NAME`, jalankan lagi.
Hapus sel bertanda `HAPUS saat lomba`.

**Data demo:** SmSA (`sentiment`, 3 kelas). Bentuk lain diuji hanya dengan mengubah Konfigurasi:
`classification` (emot, 5 kelas), `ujaran_kebencian` dengan `TARGET="HS"` (biner), `pair` (wrete)
dengan `TEXT_COLS=["sent_A", "sent_B"]`.

In [ ]:
import html
import os
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold, StratifiedKFold
from sklearn.model_selection import cross_val_predict
from sklearn.naive_bayes import ComplementNB
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline, make_union
from sklearn.svm import LinearSVC

## Konfigurasi
Semua yang berubah antar soal ada di sel ini dan sel Metrik. `OOF_FOLDER` dipakai bersama notebook
lain (`finetune_oof`, `ensemble_oof`): `oof_<MODEL_NAME>.npy` dan `test_<MODEL_NAME>.npy`.

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/sentiment"          # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/smsa_train.csv"                    # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/smsa_test.csv"                      # GANTI saat lomba
OUTPUT_FOLDER = "../../../outputs/text/baseline_oof"
OOF_FOLDER = "../../../outputs/text/oof"
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"     # GANTI saat lomba: file panitia
SUBMISSION_NAME = "submission.csv"                             # GANTI saat lomba bila diminta lain
ANSWER_CSV = f"{DATA_FOLDER}/smsa_test_labels.csv"             # HAPUS saat lomba (kunci demo)

ID_COL = "id"                    # GANTI saat lomba
TEXT_COLS = ["text"]             # GANTI saat lomba: lebih dari satu kolom = digabung dengan spasi
TARGET = "label"                 # GANTI saat lomba
SUBMIT_COL = TARGET              # GANTI saat lomba bila kolom prediksi di sample bernama lain
GROUP_COL = None                 # GANTI: kolom penulis/entitas untuk GroupKFold (None = tidak)
DATE_COL = None                  # GANTI: kolom tanggal, hanya untuk alt split waktu

N_FOLDS = 5
SEED = 42
MODEL_NAME = "tfidf_svc"         # GANTI tiap variasi (nama file OOF, jangan menimpa model lain)
DROP_DUPLICATES = False          # True = buang teks+label kembar dari data latih tiap fold
SLANG_CSV = "../../../data_latihan/preprocessing/kamus_alay_salsabila.csv"   # alt kamus alay

## Metrik
Dipakai untuk skor fold, pengali kelas, dan cek akhir. Semua di dalam memakai indeks kelas 0..K-1
(urut `np.unique` label train). Ganti baris `return` bila metrik panitia lain.

In [ ]:
METRIC_NAME = "f1_macro"         # GANTI saat lomba, ikut baris return di bawah


def score(truth, predicted):
    return f1_score(truth, predicted, average="macro")
    # return accuracy_score(truth, predicted)                            # metrik akurasi
    # return f1_score(truth, predicted, average="weighted")              # F1 berbobot
    # return f1_score(truth, predicted, average="micro")                 # F1 mikro
    # return f1_score(truth, predicted, pos_label=1)     # biner: F1 kelas ke-2 (indeks 1)

## Baca data
Kolom teks digabung jadi satu (`train_text`, `test_text`). Teks kosong/NaN jadi string kosong.

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)


def join_text(table):
    joined = table[TEXT_COLS[0]].fillna("").astype(str)
    for column in TEXT_COLS[1:]:
        joined = joined + " " + table[column].fillna("").astype(str)
    return joined.str.strip()


train_text = join_text(train)
test_text = join_text(test)
print("train:", train.shape, "| test:", test.shape)
missing_counts = train[TEXT_COLS + [TARGET]].isna().sum()
print("NaN di kolom teks/target:", missing_counts.to_dict())
train[[ID_COL] + TEXT_COLS + [TARGET]].head()

## Buat sample_submission demo (HAPUS saat lomba)
Data latihan tidak punya `sample_submission`. Saat lomba file panitia dipakai (`SAMPLE_CSV`);
sel ini tidak menimpa file di luar `OUTPUT_FOLDER`.

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
demo_sample[SUBMIT_COL] = train[TARGET].iloc[0]
if SAMPLE_CSV.startswith(OUTPUT_FOLDER):
    demo_sample.to_csv(SAMPLE_CSV, index=False)
    print("sample demo ditulis ke", SAMPLE_CSV)
else:
    print("SAMPLE_CSV bukan file demo: dibiarkan apa adanya")

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
print("sample_submission:", sample.shape, sample.columns.tolist())
print("id test = id sample:", set(test[ID_COL]) == set(sample[ID_COL]))
sample.head(3)

## Bersihkan teks
Aktif: `html.unescape`, huruf kecil, URL dan mention jadi token. Emoji dibiarkan (char n-gram
memakainya). Stopword dan stemming sengaja tidak dipakai (negasi bisa hilang).

In [ ]:
def clean_text(text):
    text = html.unescape(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " urltoken ", text)
    text = re.sub(r"@\w+", " mentiontoken ", text)
    return " ".join(text.split())


print(clean_text("Mantap murah!! Cek https://contoh.com @admin 😀"))
train_text = train_text.apply(clean_text)
test_text = test_text.apply(clean_text)

In [ ]:
def squeeze_repeats(text):
    return re.sub(r"(\w)\1{2,}", r"\1\1", text)      # "bagusssss" -> "baguss"


def normalize_slang(text):
    words = []
    for word in text.split():
        words.append(SLANG_MAP.get(word, word))
    return " ".join(words)


# Alternatif: hapus '#' di blok yang dipakai (sel ini dijalankan sesudah sel Bersihkan).
print(squeeze_repeats("bagusssss bangettt"))
# train_text = train_text.apply(squeeze_repeats)    # medsos: huruf berulang
# test_text = test_text.apply(squeeze_repeats)
# slang_table = pd.read_csv(SLANG_CSV)              # kamus alay -> baku (hanya Indonesia)
# SLANG_MAP = dict(zip(slang_table["slang"], slang_table["baku"]))
# train_text = train_text.apply(normalize_slang)
# test_text = test_text.apply(normalize_slang)

## Temuan (baca ini sebelum memilih split dan model)
Hasilnya jadi daftar `findings`, ditulis juga ke `temuan.txt` (bahan slide jebakan data).

In [ ]:
findings = []
class_counts = train[TARGET].value_counts()
findings.append(f"kelas: {len(class_counts)} | terbanyak {class_counts.iloc[0]} | "
                f"tersedikit {class_counts.iloc[-1]} ({class_counts.index[-1]})")
rare = class_counts[class_counts < 10]
if len(rare) > 0:
    findings.append(f"PERHATIAN: kelas dengan < 10 contoh: {rare.to_dict()}")
word_counts = train_text.str.split().str.len()
long_share = (word_counts > 200).mean()
findings.append(f"panjang kata: median {word_counts.median():.0f} | maks {word_counts.max()} | "
                f"p95 {word_counts.quantile(0.95):.0f} | > 200 kata: {long_share:.1%}")
findings.append(f"teks kosong: train {(train_text == '').sum()} | test {(test_text == '').sum()}")
findings.append(f"label kosong di train: {train[TARGET].isna().sum()}")
findings.append(f"id test kembar: {test[ID_COL].duplicated().sum()}")
figure, axes = plt.subplots(1, 2, figsize=(10, 3))
class_counts.plot.bar(ax=axes[0], title="Baris per kelas")
word_counts.clip(upper=300).hist(bins=50, ax=axes[1])
axes[1].set_title("Jumlah kata per teks (dipotong 300)")
plt.show()

In [ ]:
duplicated_rows = train_text.duplicated(keep=False)
labels_per_text = train.groupby(train_text)[TARGET].nunique()
conflict_texts = labels_per_text[labels_per_text > 1]
conflict_rows = train_text.isin(conflict_texts.index).sum()
findings.append(f"teks kembar di train: {duplicated_rows.sum()} baris "
                f"({train_text.duplicated().sum()} bisa dibuang)")
findings.append(f"kembar beda label: {len(conflict_texts)} teks / {conflict_rows} baris")
if conflict_rows > 0:
    findings.append("PERHATIAN: label berisik atau data ambigu, cek contoh di bawah")
exact_in_train = test_text.isin(set(train_text))
findings.append(f"teks test yang persis ada di train: {exact_in_train.sum()} dari {len(test_text)}")
print(conflict_texts.head(3))
if GROUP_COL is not None:
    shared_groups = set(train[GROUP_COL]) & set(test[GROUP_COL])
    findings.append(f"grup {GROUP_COL} yang ada di train dan test: {len(shared_groups)}")
if DATE_COL is not None:
    findings.append(f"tanggal train {train[DATE_COL].min()} s/d {train[DATE_COL].max()} | "
                    f"test {test[DATE_COL].min()} s/d {test[DATE_COL].max()}")

In [ ]:
SIMILAR_LEVEL = 0.9   # GANTI: 0.8 = lebih longgar, 0.95 = hampir persis
char_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                                  sublinear_tf=True)
train_chars = char_vectorizer.fit_transform(train_text)
checked_test = test_text.head(5000)
neighbors = NearestNeighbors(n_neighbors=1, metric="cosine").fit(train_chars)
distances, nearest = neighbors.kneighbors(char_vectorizer.transform(checked_test))
similarity = 1 - distances[:, 0]
near_rows = np.where((similarity >= SIMILAR_LEVEL) & (similarity < 0.999))[0]
findings.append(f"test hampir kembar train (sim >= {SIMILAR_LEVEL}, bukan persis): {len(near_rows)}"
                f" dari {len(checked_test)}")
for row in near_rows[:3]:
    print(round(similarity[row], 3), "|", checked_test.iloc[row][:60], "|",
          train_text.iloc[nearest[row, 0]][:60], "|", train[TARGET].iloc[nearest[row, 0]])

In [ ]:
adv_texts = pd.concat([train_text, test_text], ignore_index=True)
is_test = np.r_[np.zeros(len(train_text)), np.ones(len(test_text))]
adv_pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(C=1, max_iter=500))
adv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
adv_proba = cross_val_predict(adv_pipe, adv_texts, is_test, cv=adv_folds, method="predict_proba")
adv_auc = roc_auc_score(is_test, adv_proba[:, 1])
adv_pipe.fit(adv_texts, is_test)
adv_words = adv_pipe[0].get_feature_names_out()
top_words = []
for word_index in np.argsort(adv_pipe[-1].coef_[0])[-8:]:
    top_words.append(adv_words[word_index])
findings.append(f"adversarial AUC train vs test: {adv_auc:.3f} (0.5 = sama, > 0.65 = test beda)")
findings.append(f"kata paling 'khas test': {top_words}")

In [ ]:
for finding in findings:
    print("-", finding)
with open(f"{OUTPUT_FOLDER}/temuan.txt", "w", encoding="utf-8") as report:
    report.write("\n".join(findings))

## Split (k-fold)
Aturan format OOF: `StratifiedKFold` acak seed `SEED`, urutan baris train apa adanya. Ganti
`splitter` hanya bila Temuan menyuruh (penulis sama di train dan test → Group, data berurutan
waktu → split waktu). Split waktu hanya mengisi OOF untuk 20% terbaru, jadi OOF tidak disimpan.

In [ ]:
classes = np.unique(train[TARGET])
y = np.searchsorted(classes, train[TARGET].to_numpy())
groups = train.get(GROUP_COL)      # None bila GROUP_COL = None
if groups is not None:
    groups = groups.fillna("kosong")   # grup kosong dijadikan satu grup


def time_folds(dates, holdout=0.2):
    order = np.argsort(pd.to_datetime(dates).to_numpy(), kind="stable")
    cut = int(len(order) * (1 - holdout))
    return [(order[:cut], order[cut:])]


splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
# splitter = GroupKFold(n_splits=N_FOLDS)     # penulis/entitas sama di train & test: isi GROUP_COL
# splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)  # + timpang
folds = list(splitter.split(train_text, y, groups))
split_name = type(splitter).__name__
# folds = time_folds(train[DATE_COL], 0.2)    # data berwaktu: validasi = 20% terbaru, isi DATE_COL
# split_name = "time_split"
val_sizes = []
for _, val_index in folds:
    val_sizes.append(len(val_index))
print("kelas:", list(classes), "| fold:", len(folds), "| ukuran validasi:", val_sizes)

## Model
Pilih SATU `FEATURES` dan SATU `CLASSIFIER` (hapus `#` di baris yang dipakai, beri `#` pada
yang lain).
Model yang tidak punya `predict_proba` (Ridge) diubah ke probabilitas lewat softmax skor.

In [ ]:
FEATURES = make_union(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
)
# FEATURES = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)    # kata saja, cepat
# FEATURES = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2)   # tahan typo/alay

CLASSIFIER = CalibratedClassifierCV(LinearSVC(C=0.5, class_weight="balanced"), cv=3)   # SVM + proba
# CLASSIFIER = LogisticRegression(C=10, class_weight="balanced", max_iter=1000)   # proba asli
# CLASSIFIER = ComplementNB(alpha=0.3)                       # kelas timpang, data kecil
# CLASSIFIER = SGDClassifier(loss="modified_huber", class_weight="balanced",
#                            random_state=SEED)              # data sangat besar
# CLASSIFIER = RidgeClassifier(alpha=1.0, class_weight="balanced")   # cepat, proba = softmax skor
PIPE = make_pipeline(FEATURES, CLASSIFIER)

In [ ]:
def to_proba(pipe, texts):
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(texts)
    scores = pipe.decision_function(texts)
    if scores.ndim == 1:
        scores = np.column_stack([-scores, scores])
    return softmax(scores, axis=1)


def fit_predict(fit_texts, fit_y, texts_list):
    pipe = clone(PIPE)
    pipe.fit(fit_texts, fit_y)
    outputs = []
    for texts in texts_list:
        outputs.append(to_proba(pipe, texts))
    return outputs


print(PIPE)

In [ ]:
def log_count_ratio(counts, is_class, alpha=1.0):
    positive = np.asarray(counts[is_class].sum(axis=0)).ravel() + alpha
    negative = np.asarray(counts[~is_class].sum(axis=0)).ravel() + alpha
    return np.log((positive / positive.sum()) / (negative / negative.sum()))


def fit_predict_nbsvm(fit_texts, fit_y, texts_list):
    counter = CountVectorizer(ngram_range=(1, 2), min_df=2, binary=True)
    counts = counter.fit_transform(fit_texts)
    outputs = []
    for texts in texts_list:
        outputs.append(np.zeros((len(texts), len(np.unique(fit_y)))))
    for class_number in np.unique(fit_y):
        ratio = log_count_ratio(counts, fit_y == class_number)
        model = LogisticRegression(C=4, max_iter=1000)
        model.fit(counts.multiply(ratio).tocsr(), fit_y == class_number)
        for output, texts in zip(outputs, texts_list):
            counts_test = counter.transform(texts)
            features = counts_test.multiply(ratio).tocsr()
            output[:, class_number] = model.predict_proba(features)[:, 1]
    return outputs


# NB-SVM (rasio log-count NB + LogReg per kelas): sering menang di sentimen, data kecil-sedang.
# fit_predict = fit_predict_nbsvm    # hapus '#' untuk NB-SVM (FEATURES/CLASSIFIER diabaikan)
print("fit_predict aktif:", fit_predict.__name__)

## Latih k-fold
Tiap fold: latih di bagian latih, prediksi bagian validasi (OOF) dan seluruh test. Probabilitas
test = rata-rata model tiap fold (tidak dilatih ulang di semua data).

In [ ]:
def drop_repeats(fit_index):
    pairs = pd.DataFrame({"text": train_text.iloc[fit_index].to_numpy(), "y": y[fit_index]})
    return fit_index[~pairs.duplicated().to_numpy()]


oof = np.zeros((len(train), len(classes)), dtype="float32")
test_proba = np.zeros((len(test), len(classes)), dtype="float32")
fold_scores = []
fold_seconds = []
for number, (fit_index, val_index) in enumerate(folds):
    start = time.time()
    if DROP_DUPLICATES:
        fit_index = drop_repeats(fit_index)
    val_proba, fold_test = fit_predict(train_text.iloc[fit_index], y[fit_index],
                                       [train_text.iloc[val_index], test_text])
    oof[val_index] = val_proba
    test_proba = test_proba + fold_test / len(folds)
    fold_scores.append(score(y[val_index], val_proba.argmax(axis=1)))
    fold_seconds.append(round(time.time() - start, 1))
    print(f"fold {number}: {METRIC_NAME} {fold_scores[-1]:.4f} | {fold_seconds[-1]} detik")
print(f"rata-rata fold: {np.mean(fold_scores):.4f} +- {np.std(fold_scores):.4f}")

## Evaluasi OOF
Skor dihitung di semua baris train yang punya prediksi OOF (di split waktu: hanya 20% terbaru).

In [ ]:
filled = oof.sum(axis=1) > 0
oof_pred = oof.argmax(axis=1)
oof_score = score(y[filled], oof_pred[filled])
print(f"{METRIC_NAME} OOF ({filled.sum()} baris): {oof_score:.4f}")
print(classification_report(y[filled], oof_pred[filled], target_names=classes.astype(str)))
ConfusionMatrixDisplay.from_predictions(y[filled], oof_pred[filled], display_labels=classes,
                                        normalize="true", xticks_rotation=45)
plt.show()

## Pengali kelas (F1 makro)
Mencari pengali per kelas pada OOF (coordinate ascent) lalu dipakai ke probabilitas test. Dipakai
hanya bila naik ≥ `MIN_GAIN` (dihitung di OOF yang sama, jadi sedikit optimistis).

In [ ]:
def optimize_class_weights(proba, truth, rounds=2):
    weights = np.ones(proba.shape[1])
    best = score(truth, proba.argmax(axis=1))
    for _ in range(rounds):
        for class_number in range(len(weights)):
            for candidate in [0.5, 0.75, 1, 1.5, 2, 3, 5, 8]:
                trial = weights.copy()
                trial[class_number] = candidate
                trial_score = score(truth, (proba * trial).argmax(axis=1))
                if trial_score > best:
                    best, weights = trial_score, trial
    return weights, best


def best_threshold(proba_positive, truth):
    best_cut, best_value = 0.5, -1.0
    for cut in np.arange(0.05, 0.96, 0.01):
        value = score(truth, (proba_positive >= cut).astype(int))
        if value > best_value:
            best_cut, best_value = cut, value
    return best_cut, best_value


print("fungsi pengali siap")

In [ ]:
class_weights, weighted_score = optimize_class_weights(oof[filled], y[filled])
# cut, weighted_score = best_threshold(oof[filled, 1], y[filled])   # alt biner: ambang kelas ke-2
# class_weights = np.array([cut, 1 - cut])                          # (setara pengali)
MIN_GAIN = 0.002   # pengali dipakai hanya bila skor OOF naik sekurangnya ini
gain = weighted_score - oof_score
print("pengali:", class_weights.round(2), f"| skor OOF {weighted_score:.4f} (naik {gain:.4f})")
if gain < MIN_GAIN:
    class_weights = np.ones(len(classes))
    print("kenaikan kecil: pengali tidak dipakai")

## Simpan OOF dan catat percobaan
OOF disimpan bila lengkap (semua baris train terisi). `percobaan.csv` bertambah satu baris tiap
run.

In [ ]:
if filled.all():
    np.save(f"{OOF_FOLDER}/oof_{MODEL_NAME}.npy", oof)
    np.save(f"{OOF_FOLDER}/test_{MODEL_NAME}.npy", test_proba)
    print("OOF dan test disimpan:", MODEL_NAME)
else:
    print("OOF tidak lengkap (split waktu?): tidak disimpan")
log_path = f"{OOF_FOLDER}/percobaan.csv"
note = (f"{split_name} | pengali {weighted_score:.4f} | fold {np.mean(fold_scores):.4f} | "
        f"detik/fold {fold_seconds}")
if split_name != "StratifiedKFold":
    print("PERHATIAN: fold bukan StratifiedKFold standar, OOF ini tidak sefold dengan model lain")
row = {"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "notebook": "baseline_oof", "model": MODEL_NAME,
       "metrik": METRIC_NAME, "skor_cv": round(oof_score, 4), "detik": round(sum(fold_seconds), 1),
       "catatan": note}
pd.DataFrame([row]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
history = pd.read_csv(log_path)
print(history.tail(5).to_string())

## Submission
Kolom id diambil dari `sample_submission`; prediksi dipetakan lewat id.

In [ ]:
test_pred = (test_proba * class_weights).argmax(axis=1)
pred_by_id = pd.Series(classes[test_pred], index=test[ID_COL])

submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUBMIT_COL] = sample[ID_COL].map(pred_by_id)
predicted_share = submission[SUBMIT_COL].value_counts(normalize=True)
print("sebaran prediksi test:", predicted_share.round(3).to_dict())
# soal minta probabilitas per kelas: hapus '#' tiga baris ini (sample panitia harus berkolom kelas)
# proba_by_id = pd.DataFrame(test_proba, columns=classes.astype(str), index=test[ID_COL])
# proba_by_id = proba_by_id.reindex(sample[ID_COL]).reset_index(drop=True)
# submission = pd.concat([submission[[ID_COL]], proba_by_id], axis=1)
submission.to_csv(f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}", index=False)
submission.head()

## Cek submission terhadap sample_submission
Salinan validator T3 (`arsenal/umum/simpan_muat` #15 dan #16): hanya membaca dan mencetak. Hasil
akhir harus `MASALAH: 0`. Dicek: kolom dan urutannya, jumlah baris, id hilang / asing / kembar /
urutan beda, nilai kosong, tipe kolom ("3" vs "3.0"), label di luar `ALLOWED`.

In [ ]:
SUBMISSION_CSV = f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}"
ID_COLUMN = ID_COL

sample = pd.read_csv(SAMPLE_CSV)
written = pd.read_csv(SUBMISSION_CSV)
problems = []


def report(failed, text):
    if failed:
        problems.append(text)


wanted = sample[ID_COLUMN]
ids = written.get(ID_COLUMN, pd.Series(dtype=wanted.dtype))
lost = set(wanted) - set(ids)
extra = set(ids) - set(wanted)
kept = ids[ids.isin(wanted)].drop_duplicates().tolist()
expected = wanted[wanted.isin(ids)].tolist()
report(written.columns.tolist() != sample.columns.tolist(), "kolom / urutan kolom beda")
report(len(written) != len(sample), f"jumlah baris {len(written)}, sample {len(sample)}")
report(lost, f"id hilang {len(lost)}, mis. {sorted(lost)[:3]}")
report(extra, f"id asing {len(extra)}, mis. {sorted(extra)[:3]}")
report(ids.duplicated().any(), f"id kembar: {ids.duplicated().sum()} baris")
report(kept != expected, "urutan id beda dari sample")

In [ ]:
LABEL_COLUMNS = [SUBMIT_COL]     # kolom isi kelas (soal proba per kelas: isi [] untuk lewati cek)
ALLOWED = list(classes)          # label sah = label train, [] = lewati cek

for column in sample.columns:
    if column in written.columns:
        empty = written[column].isna().sum()
        report(empty, f"{column}: {empty} nilai kosong")
        report(written[column].dtype != sample[column].dtype,
               f"{column}: tipe {written[column].dtype}, sample {sample[column].dtype}")
for column in LABEL_COLUMNS:
    if ALLOWED and column in written.columns:
        foreign = set(written[column].dropna()) - set(ALLOWED)
        report(foreign, f"{column}: label di luar ALLOWED {sorted(foreign)[:5]}")
        print(written[column].value_counts())   # semua satu kelas -> curigai model
for problem in problems:
    print("-", problem)
print("MASALAH:", len(problems))

## Cek dengan kunci jawaban (HAPUS saat lomba)
Hanya untuk data latihan yang punya kunci (`ANSWER_CSV` di Konfigurasi). Hapus sel ini dan
baris `ANSWER_CSV` saat lomba.

In [ ]:
answers = pd.read_csv(ANSWER_CSV)
truth_by_id = pd.Series(answers[TARGET].to_numpy(), index=answers[ID_COL])
truth_index = np.searchsorted(classes, truth_by_id.reindex(submission[ID_COL]).to_numpy())
pred_index = np.searchsorted(classes, submission[SUBMIT_COL].to_numpy())
print(f"{METRIC_NAME} di test (kunci): {score(truth_index, pred_index):.4f}")
print(f"{METRIC_NAME} OOF train: {oof_score:.4f}")